In [3]:
import streamlit as st
import time
import pandas as pd
import numpy as np
import requests
import json
import datetime



In [4]:
def mongol_bank_khansh(date: str) -> pd.DataFrame:
    # 1. Огнооны форматыг шалгах
    try:
        datetime.strptime(date, "%Y-%m-%d")
    except ValueError:
        raise ValueError("Invalid date format. Expected YYYY-MM-DD")

    url = f"https://www.mongolbank.mn/mn/currency-rates/data?startDate={date}&endDate={date}"

    try:
     
        response = requests.post(url) 
        
        if response.status_code != 200:
            raise Exception(f"Хүсэлт амжилтгүй: {response.status_code}")
            
        data = response.json()
        
        if not data.post("success", False):
            return "API дуудахад алдаа гарлаа. Түр хүлээгээд дахиад оролдоорой."

        if "data" not in data or not data["data"]:
            return "Тухайн өдрийн ханшийн мэдээлэл байхгүй байна."

        return pd.DataFrame(data["data"])
        
    except Exception as e:
        return f"Алдаа гарлаа: {str(e)}


SyntaxError: unterminated f-string literal (detected at line 28) (3428350865.py, line 28)

In [5]:
OPS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.Pow: operator.pow,
    ast.USub: operator.neg
}

def eval_expr(expr: str):
    def _eval(node):
        
        if isinstance(node, ast.Constant): # Modern Python (3.8+)
            return node.value
        elif isinstance(node, ast.Num): # Older Python
            return node.n
        elif isinstance(node, ast.BinOp):
            return OPS[type(node.op)](_eval(node.left), _eval(node.right))
        elif isinstance(node, ast.UnaryOp):
            return OPS[type(node.op)](_eval(node.operand))
        else:
            raise ValueError("Unsupported expression")

    try:
        parsed = ast.parse(expr, mode='eval')
        return _eval(parsed.body)
    except ZeroDivisionError:
        return "Division by zero"
    except Exception:
        return "Invalid expression"
def handle_user_query(user_query: str):
    if not isinstance(user_query, str):
        return "Invalid input"

    query = user_query.strip()
    query_lower = query.lower()

    # 1. Холбоо барих мэдээлэл
    if any(word in query_lower for word in ["утас", "contact", "холбоо барих"]):
        return "Манай холбоо барих утасны дугаар: 99887766"

    # 2. Байршил
    if any(word in query_lower for word in ["байршил", "location"]):
        return "Манай байршил: Galaxy tower 7 давхар, 705 тоот"

    # 3. Огноо мөн эсэхийг шалгах (YYYY-MM-DD) -> Ханш дуудах
    if re.match(r"^\d{4}-\d{2}-\d{2}$", query):
        return mongol_bank_khansh(query)

    # 4. Математик илэрхийлэл мөн эсэхийг шалгах
    if re.match(r"^[0-9+\-*/().\s^]+$", query):
        return eval_expr(query)

    # 5. Бусад тохиолдолд ирсэн текстийг буцаах
    return user_query
        

NameError: name 'ast' is not defined